##### WebScraping a wikipedia webpage on covid19 and its variants

In [1]:
import os
import re
from dotenv import load_dotenv
import bs4

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
load_dotenv('../../.env')

C:\Users\omole\AppData\Local\Temp\ipykernel_2180\2907290677.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [2]:
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')
os.environ['USER_AGENT'] = 'MYCovid19-Agent'

In [3]:
url = "https://www.nih.gov/news-events/nih-research-matters/depression-may-stall-development-new-neurons"
webloader = WebBaseLoader(url)
docs = webloader.load()

In [4]:
for doc in docs:
    doc.page_content = re.sub(r'\n+', '\n', doc.page_content)
    doc.page_content = '\n'.join([line.strip() for line in doc.page_content.split('\n') if line.strip()])
docs[0]

Document(metadata={'source': 'https://www.nih.gov/news-events/nih-research-matters/depression-may-stall-development-new-neurons', 'title': 'Depression may stall development of new neurons | National Institutes of Health (NIH)', 'description': 'Researchers discovered how a part of the brain important for memory struggles to make new neurons in people with depression.', 'language': 'en'}, page_content="Depression may stall development of new neurons | National Institutes of Health (NIH)\nSkip to main content\nAn official website of the United States government\nHere’s how you know\nHere’s how you know\nOfficial websites use .gov\nA .gov website belongs to an official government organization in the United States.\nSecure .gov websites use HTTPS\nA lock\n(LockA locked padlock)\nor https:// means you’ve safely connected to the .gov website. Share sensitive information only on official, secure websites.\nNIH Websites Are Changing\nNIH is moving the launch of its new website to the fall to en

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
final_chunk = text_splitter.split_documents(docs)

In [6]:
from langchain_community.vectorstores import Chroma

In [7]:
embeddings = OllamaEmbeddings(model='nomic-embed-text')

vector_store = Chroma.from_documents(final_chunk, embeddings)
retriever = vector_store.as_retriever(search_kwargs={'k':4})

In [8]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='medllama2', temperature=0.2)

system_prompt = (
        "You are an advanced AI medical assistant. Answer the user's question based strictly "
        "on the provided context below. If you do not know the answer or if it's not in the context, "
        "say 'I cannot find that information in the provided records.' Do not make things up.\n\n"
        "Context:\n{context}"
    )


In [12]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', 'Question : {input}')
    ]
)

In [13]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
chain = prompt|llm|output_parser

In [15]:
input_text = input('What\'s up, how may I help you today???')
if input_text:
    print(chain.invoke(
        {
            'input': input_text,
            'context' : ''
         }
    ))

Depression is a serious mental illness characterized by persistent feelings of sadness and hopelessness. It can also include a loss of interest in activities, changes in appetite and sleep, and even suicidal thoughts. Depression can be treated with therapy and medication. If you think you might be experiencing depression, please consult a mental health professional.
